# P5.1 CPU-only platform baseline

No bitstream download, DMA, MMIO or RF operations. Actual display reference: 800x480.

In [1]:
import os,sys,json,time,platform,hashlib
from pathlib import Path
os.environ['XILINX_XRT']='/usr'
import numpy as np
import pynq
from pynq import Overlay,Bitstream,allocate
root=Path('/home/xilinx/sentinel')
sys.path.insert(0,str(root))
measure=root/'measurements'
measure.mkdir(exist_ok=True)
result={'kernel':platform.platform(),'python':sys.version,'pynq':pynq.__version__,
        'lcd_resolution':[800,480],'lcd_interface':'SSD1963 J22 8080'}
print(json.dumps(result,indent=2))
print(Path('/proc/meminfo').read_text().splitlines()[0])
print(Path('/proc/cpuinfo').read_text())
HARDWARE_VALIDATED=False  # Keep false until the interrupted platform test is diagnosed.

{
  "kernel": "Linux-5.4.0-xilinx-v2020.2-armv7l-with-glibc2.29",
  "python": "3.8.2 (default, Mar 13 2020, 10:14:16) \n[GCC 9.3.0]",
  "pynq": "2.7.0",
  "lcd_resolution": [
    800,
    480
  ],
  "lcd_interface": "SSD1963 J22 8080"
}
MemTotal:        1026040 kB
processor	: 0
model name	: ARMv7 Processor rev 0 (v7l)
BogoMIPS	: 766.66
Features	: half thumb fastmult vfp edsp neon vfpv3 tls vfpd32 
CPU implementer	: 0x41
CPU architecture: 7
CPU variant	: 0x3
CPU part	: 0xc09
CPU revision	: 0

processor	: 1
model name	: ARMv7 Processor rev 0 (v7l)
BogoMIPS	: 766.66
Features	: half thumb fastmult vfp edsp neon vfpv3 tls vfpd32 
CPU implementer	: 0x41
CPU architecture: 7
CPU variant	: 0x3
CPU part	: 0xc09
CPU revision	: 0

Hardware	: Xilinx Zynq Platform
Revision	: 0003
Serial		: 0000000000000000



In [2]:
from sim.golden.cfar import cfar,alpha_for_pfa
from PIL import Image,ImageDraw
rng=np.random.default_rng(372)
x=(rng.normal(size=2048)+1j*rng.normal(size=2048)).astype(np.complex64)
p=np.rint(rng.exponential(2**18,2048)).astype(np.uint32)
alpha=alpha_for_pfa(.001)
def bench(function,repeats):
    for _ in range(5): function()
    elapsed=[]
    for _ in range(repeats):
        start=time.perf_counter();function();elapsed.append(time.perf_counter()-start)
    return float(np.median(elapsed))
fft_s=bench(lambda:np.fft.fft(x),1000)
cfar_s=bench(lambda:cfar(p,alpha_q88=alpha),100)
points=[(i,int(80+60*np.sin(i*.07))) for i in range(720)]
waterfall=Image.fromarray(rng.integers(0,256,(200,720,3),dtype=np.uint8))
def draw_frame():
    frame=Image.new('RGB',(800,480));draw=ImageDraw.Draw(frame)
    draw.line(points,fill=(0,255,255));frame.paste(waterfall,(0,160))
    return frame
render_s=bench(draw_frame,100)
result['cpu_baseline']={'fft2048_median_s':fft_s,'fft_max_frames_per_s':1/fft_s,
                        'cfar2048_median_s':cfar_s,'pillow800x480_median_s':render_s}
print(json.dumps(result['cpu_baseline'],indent=2))

(measure/'p5_cpu_baseline.json').write_text(json.dumps(result,indent=2),encoding='utf-8')

{
  "fft2048_median_s": 0.0006743099999937385,
  "fft_max_frames_per_s": 1482.997434428209,
  "cfar2048_median_s": 0.011951670000001968,
  "pillow800x480_median_s": 0.006794262500022796
}


453

## HUMAN screen observation

Report whether eight vertical color bars appear and whether the top region changes periodically.
The display ID is the confirmed module model, not a readback: this write-only test holds RD high.
No claim of touch or ID readback is made. Save a screen photo under measurements/p5_lcd.jpg for competition evidence.
